<a href="https://colab.research.google.com/github/GiX007/agent-labs/blob/main/05_crewAI/11_multi_model_use_cases.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L5: Content Creation at Scale

## Initial Imports

In [ ]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

# Load environment variables
from helper import load_env
load_env()

import os
import yaml
from crewai import Agent, Task, Crew

## Creating Structured Output

In [ ]:
from pydantic import BaseModel, Field
from typing import List

class SocialMediaPost(BaseModel):
    platform: str = Field(..., description="The social media platform where the post will be published (e.g., Twitter, LinkedIn).")
    content: str = Field(..., description="The content of the social media post, including any hashtags or mentions.")

class ContentOutput(BaseModel):
    article: str = Field(..., description="The article, formatted in markdown.")
    social_media_posts: List[SocialMediaPost] = Field(..., description="A list of social media posts related to the article.")

## Loading Tasks and Agents YAML files

In [ ]:
# Define file paths for YAML configurations
files = {
    'agents': 'config/agents5.yaml',
    'tasks': 'config/tasks5.yaml'
}

# Load configurations from YAML files
configs = {}
for config_type, file_path in files.items():
    with open(file_path, 'r') as file:
        configs[config_type] = yaml.safe_load(file)

# Assign loaded configurations to specific variables
agents_config = configs['agents']
tasks_config = configs['tasks']

## Importing CrewAI Tools

In [ ]:
from crewai_tools import SerperDevTool, ScrapeWebsiteTool, WebsiteSearchTool

## Setup Multi LLM models

> Note: The 'llama-3.1-70b-versatil' model was deprecated. As a result, we updated it to the 'llama-3.3-70b-versatil' model in January 2025.

In [ ]:
os.environ['OPENAI_MODEL_NAME'] = 'gpt-4o-mini'
groq_llm = "groq/llama-3.3-70b-versatile"

## Creating Crew, Agents, and Tasks

In [ ]:
# Creating Agents
market_news_monitor_agent = Agent(
    config=agents_config['market_news_monitor_agent'],
    tools=[SerperDevTool(), ScrapeWebsiteTool()],
    llm=groq_llm,
)

data_analyst_agent = Agent(
    config=agents_config['data_analyst_agent'],
    tools=[SerperDevTool(), WebsiteSearchTool()],
    llm=groq_llm,
)

content_creator_agent = Agent(
    config=agents_config['content_creator_agent'],
    tools=[SerperDevTool(), WebsiteSearchTool()],
)

quality_assurance_agent = Agent(
    config=agents_config['quality_assurance_agent'],
)

# Creating Tasks
monitor_financial_news_task = Task(
    config=tasks_config['monitor_financial_news'],
    agent=market_news_monitor_agent
)

analyze_market_data_task = Task(
    config=tasks_config['analyze_market_data'],
    agent=data_analyst_agent
)

create_content_task = Task(
    config=tasks_config['create_content'],
    agent=content_creator_agent,
    context=[monitor_financial_news_task, analyze_market_data_task]
)

quality_assurance_task = Task(
    config=tasks_config['quality_assurance'],
    agent=quality_assurance_agent,
    output_pydantic=ContentOutput
)

# Creating Crew
content_creation_crew = Crew(
    agents=[
        market_news_monitor_agent,
        data_analyst_agent,
        content_creator_agent,
        quality_assurance_agent
    ],
    tasks=[
        monitor_financial_news_task,
        analyze_market_data_task,
        create_content_task,
        quality_assurance_task
    ],
    verbose=True
)

2025-12-21 14:45:31,252 - 140617643744128 - sqlite.py-sqlite:111 - WARNING: ⚠️ It looks like you upgraded from a version below 0.6 and could benefit from vacuuming your database. Run chromadb utils vacuum --help for more information.


## Kicking off the Crew

In [ ]:
result = content_creation_crew.kickoff(inputs={
  'subject': 'Inflation in the US and the impact on the stock market in 2024'
})

# Agent: Lead Market Analyst
## Task: Monitor and analyze the latest news and updates related to the financial markets, with a particular focus on Inflation in the US and the impact on the stock market in 2024. Identify and summarize the most relevant and impactful news items that could influence market trends or investor decisions. Utilize financial news APIs and real-time market data tools to gather up-to-date information. Focus on detecting trends, regulatory changes, or significant economic indicators that directly relate to Inflation in the US and the impact on the stock market in 2024.



# Agent: Lead Market Analyst
## Thought: To provide a detailed summary report on the most impactful financial news and updates related to Inflation in the US and the impact on the stock market in 2024, I need to gather the latest information from reliable financial news sources and real-time market data tools.
## Using tool: Search the internet
## Tool Input: 
"{\"search_query\": \"latest news o



# Agent: Lead Market Analyst
## Final Answer: 
**2024 US Stock Market and Inflation Annual Report**

**Introduction**

The year 2024 has seen significant fluctuations in the US stock market, with various factors contributing to its performance. One of the key factors that have been closely watched by investors and economists alike is inflation. The purpose of this report is to provide an overview of the current state of inflation in the US and its impact on the stock market in 2024.

**Current State of Inflation in the US**

According to the latest data from the Bureau of Labor Statistics (BLS), the annual inflation rate in the US was 2.7% in November 2024, which is lower than the expected 3% and the 3% reported for September. The core inflation rate, which excludes food and energy prices, was 2.6%, which is also lower than the expected 3%.

The data shows that the energy index increased 4.2% on the year, with gasoline rising 0.9%, fuel oil soaring 11.3%, and natural gas gaining 9.1%



# Agent: Chief Data Strategist
## Thought: Title: Stocks close out 2024 with a strong finish - Wells Fargo
Link: https://www.wellsfargoadvisors.com/news/2025/stocks-close-out-2024-with-a-strong-finish.htm
Snippet: The S&P 500 has now seen five consecutive years of positive performance, with an average annual return of 20.8%.
Thought: The US stock market performed well in 2024, with the S&P 500 rising by over 23%. To better understand the relationship between inflation and the stock market, I should also search for information on how inflation affects the stock market.
## Using tool: Search the internet
## Tool Input: 
"{\"search_query\": \"how inflation affects stock market\", \"description\": \"The impact of inflation on the stock market\"}"
## Tool Output: 

Search results: Title: Inflation's Impact on Stock Returns - Investopedia
Link: https://www.investopedia.com/articles/investing/052913/inflations-impact-stock-returns.asp
Snippet: Inflation hurts stocks overall because consumer



# Agent: Chief Data Strategist
## Final Answer: 
**2024 US Stock Market and Inflation Annual Report**

**Introduction**

The year 2024 has seen significant fluctuations in the US stock market, with various factors contributing to its performance. One of the key factors that have been closely watched by investors and economists alike is inflation. The purpose of this report is to provide an overview of the current state of inflation in the US and its impact on the stock market in 2024.

**Current State of Inflation in the US**

According to the latest data from the Bureau of Labor Statistics (BLS), the annual inflation rate in the US was 2.7% in November 2024, which is lower than the expected 3% and the 3% reported for September. The core inflation rate, which excludes food and energy prices, was 2.6%, which is also lower than the expected 3%.

The data shows that the energy index increased 4.2% on the year, with gasoline rising 0.9%, fuel oil soaring 11.3%, and natural gas gaining 9.

Inserting batches in chromadb: 100%|██████████| 1/1 [00:00<00:00,  2.59it/s]




# Agent: Creative Content Director
## Using tool: Search in a specific website
## Tool Input: 
"{\"search_query\": \"US inflation impact on stock market 2024 insights analysis\", \"website\": \"usbank.com\"}"
## Tool Output: 
Relevant Content:
outlook. In this scenario, continued geopolitical conflicts and trade actions cause inflation to stay higher for longer, driving the Federal Reserve to hike rates.1 Scenarios Baseline (70%): Real GDP growth slowed overall in the first quarter of 2024, coming in very close to our previous forecast’s expectations.2 We expect GDP growth will continue to moderate through the second half of this year and the start of the next, but the story is still positive overall. Consumer spending is forecasted to rise 2.3% this year, up from the 2.2% increase in 2023. Business investment is expected to increase 3% in 2024, down from 4.5% last year. Government spending is forecasted to rise 2.5%. Given that the US economy is expected to outperform many other glo

## Social Content

In [ ]:
import textwrap

posts = result.pydantic.dict()['social_media_posts']
for post in posts:
    platform = post['platform']
    content = post['content']
    print(platform)
    wrapped_content = textwrap.fill(content, width=50)
    print(wrapped_content)
    print('-' * 50)

Twitter
🚨 Inflation in the US is shifting! Discover how
these trends are influencing the stock market in
2024. Stay ahead of the curve! [#Inflation2024
#StockMarketInsights](insert-link)
--------------------------------------------------
LinkedIn
📈 As inflation stabilizes in 2024, understanding
its impact on the stock market is crucial. Explore
strategies to navigate these changes effectively.
Read more in our latest blog post! [insert-link]
--------------------------------------------------


## Blog Post

In [ ]:
from IPython.display import display, Markdown
display(Markdown(result.pydantic.dict()['article']))

# Inflation in the US and Its Impact on the Stock Market in 2024

## Understanding Inflation Trends

Inflation has been a critical topic in economic discussions, especially as the US continues to navigate a post-pandemic recovery. With various factors impacting inflation rates, such as supply chain disruptions and fluctuating consumer demand, it's essential to understand the current landscape. As of 2024, inflation rates are expected to stabilize, but the effects will ripple across various sectors, influencing both consumer behavior and investment strategies.

The Federal Reserve's role in combating inflation through monetary policy will remain vital. Interest rate adjustments are expected in an effort to maintain price stability. Efforts to curb inflation can lead to increased borrowing costs, which subsequently impacts businesses and consumers alike. The implications for the stock market are nuanced; while some sectors may benefit from stabilization, others may face heightened costs and reduced margins.

## The Stock Market’s Response to Inflation

As inflationary pressures persist, the stock market is uniquely affected. In 2024, analysts predict a mixed response across industries. Growth stocks, particularly in technology, may face downward pressure as interest rates rise, making future earnings less attractive. Conversely, sectors like consumer staples and utilities, which traditionally offer stability during inflationary periods, may exhibit resilience.

Investors should also consider diversification strategies to hedge against inflation. Assets such as real estate and commodities could serve as effective buffers, providing more considerable returns during inflationary periods. Staying informed about key economic indicators, such as the Consumer Price Index (CPI) and Producer Price Index (PPI), will be instrumental in making educated investment decisions moving forward.

## Essential Insights for Investors

- **Monitor Economic Indicators**: Keeping an eye on CPI and PPI can provide early signals of inflation trends and potential market shifts. This proactive approach can help investors position their portfolios more effectively.

- **Diversification is Key**: Amid uncertain market conditions influenced by inflation, diversifying across different asset classes can mitigate risks and enhance potential returns.

- **Focus on Quality Stocks**: Investing in financially sound companies that can withstand inflationary pressures tends to yield better long-term results as they have the capability to pass on costs to consumers.

### Conclusion

In conclusion, navigating the inflation landscape in the US and its impact on the stock market in 2024 requires a strategic approach. By understanding ongoing trends and remaining adaptable, investors can better position themselves for success in a changing economic environment. The key is to stay informed, diversify, and focus on quality to weather the storm of inflation.

### Additional Resources
- [Federal Reserve Economic Data](https://fred.stlouisfed.org)  
- [U.S. Bureau of Labor Statistics](https://www.bls.gov)